# Word2Vec

Word2Vec is a predictive model used for creating word embeddings, which are dense vector representations of words.[1] These embeddings capture semantic and syntactic relationships between words, enabling machines to understand and process natural language more effectively[2]. Word2Vec is not a single algorithm but a family of model architectures and optimizations used to learn high-quality word embeddings (vector representations of words) from raw text. Word2Vec was invented by a team of researchers at Google led by Tomas Mikolov in 2013. The original articles were published as "Efficient Estimation of Word Representations in Vector Space" and "Distributed Representations of Words and Phrases and their Compositionality."[3,4]

[1]- https://en.wikipedia.org/wiki/Word2vec

[2]- <a href="https://www.ibm.com/think/topics/word-embeddings#:~:text=Word%20embeddings%20have%20proven%20invaluable,process%20the%20semantic%20relationships%20between">https://www.ibm.com/think/topics/word-embeddings</a>

[3]- Mikolov, T., Chen, K., Corrado, G., & Dean, J. (2013). *Efficient Estimation of Word Representations in Vector Space.* arXiv:1301.3781

[4]- Mikolov, T., Sutskever, I., Chen, K., Corrado, G., & Dean, J. (2013). *Distributed Representations of Words and Phrases and their Compositionality.* NIPS 2013



### Intuitive and Logical Concept

* **Idea:** Word2Vec maps words to **dense, low-dimensional vectors** (embeddings) such that **words appearing in similar contexts are close in vector space**.
* **Analogy:** If "king" and "queen" appear in similar sentences, their vectors will be close; operations like **king - man + woman ≈ queen** become possible.
* **Contrast with BoW:** BoW is **sparse, high-dimensional, and ignores context similarity**. Word2Vec captures **semantic similarity and relations** in a compact, continuous space.

### Mathematical Background

Word2Vec has two main architectures:
- CBOW (Continuous Bag of Words)
  * Predict a **center word**$^1$ $w_t$ from its **context words**$^2$ $w_{t-m},...,w_{t-1}, w_{t+1},...,w_{t+m}$.

  * Objective: Maximize the log probability: $\mathcal{L} = \frac{1}{T} \overset{T}{\underset{t=1}{\sum}} \log P(w_t | context)$

    where: $P(w_t \mid \text{context}) = \frac{\exp({v_{w_t}'}^\top \cdot h)}{\overset{|V|}{\underset{i=1}{\sum}} \exp({v_{w_i}'}^\top \cdot h)}$

      * $h=\frac{1}{2m}\underset{j=-m}{\overset{m}{\sum}}v_{m_{t+j}}, j\not = 0$ is the average of context word vectors.
      * $|V|$ is  vocabulary size,
      * $v_{w_i}$ is the input vector for word $w_i$,
      * $v^{\prime}_w$ is the output vector.
      * $T$ represents the total number of words in the corpus.
- Skip Gram
    * Predict context words from a center word.
    * Objective:
    $\mathcal{L} = \frac{1}{T} \overset{T}{\underset{t=1}{\sum}}\ \  \underset{\begin{matrix}j=-m \\ j\not = 0\end{matrix}}{\overset{m}{\sum}}\log P(w_{t+j} | w_t)$

    where:
    $P(w_O \mid w_I) = \frac{\exp({v_{w_O}'}^\top v_{w_I})}{\sum_{w=1}^{V} \exp({v_w'}^\top v_{w_I})}$



**foot notes**

1- The center word is the target word we are currently trying to represent or predict in Word2Vec, In Skip-Gram we start from the center word and try to predict its surrounding words. In CBOW we predict the center word based on its surrounding words. Imagine reading a sentence. At each step, the word you are “focusing on” is the center word. for example in sentence "_The quick brown fox jumps over the lazy dog_", If the center word is "_fox_", we focus on "_fox_" in that step.

2- Context words are the words surrounding the center word within a fixed “window” size. They provide semantic context for learning the representation of the center word. The window size determines how many words on the left and right are considered. in the mentioned sentence if we set window size to `2`, for center word _fox_  context words (2 on each side) are "quick", "brown", "jumps", "over". So for this step:

Center word= "fox" then Context words=["quick","brown","jumps","over"] 

where

$$
P(w_t | context) = \frac{\exp(v'_{w_t}^\top \cdot h)}{\sum_{w=1}^{V} \exp(v'_w^\top \cdot h)}
$$

* $h = \frac{1}{2m} \sum_{-m \le j \le m, j \neq 0} v_{w_{t+j}}$ is the **average of context word vectors**.
* $v_w$ is the **input vector** for word $w$, $v'_w$ is the **output vector**.
* $V$ is vocabulary size.

### 3.2 Skip-Gram

* Predict **context words** from a **center word**
* Objective:

$$
\mathcal{L} = \frac{1}{T} \sum_{t=1}^{T} \sum_{-m \le j \le m, j \neq 0} \log P(w_{t+j} | w_t)
$$

* Softmax for probability:

$$
P(w_O | w_I) = \frac{\exp(v'_{w_O}^\top v_{w_I})}{\sum_{w=1}^{V} \exp(v'_w^\top v_{w_I})}
$$

* Training often uses **negative sampling** or **hierarchical softmax** for efficiency.

---

## 4. Common Applications

* **Word similarity / semantic search:** Find synonyms, analogies
* **Text classification / sentiment analysis**
* **Named Entity Recognition (NER)**
* **Machine translation / multilingual embeddings**
* **Recommendation systems** (product or content embeddings)
* **Clustering / topic modeling** using vector similarities

---

## 5. Manual Implementation (from scratch, Python)

Here’s a **very simple Skip-Gram with one-hot encoding**, without optimization tricks:

```python
import numpy as np
from collections import defaultdict

# -----------------------
# Sample corpus
# -----------------------
corpus = ["I like deep learning", "I like NLP", "I enjoy machine learning"]
corpus = [sentence.lower().split() for sentence in corpus]

# Build vocabulary
words = set([w for sentence in corpus for w in sentence])
word2idx = {w:i for i,w in enumerate(words)}
idx2word = {i:w for w,i in word2idx.items()}
V = len(words)  # vocab size

# One-hot encoding
def one_hot(word):
    vec = np.zeros(V)
    vec[word2idx[word]] = 1
    return vec

# -----------------------
# Generate training pairs (Skip-Gram, window=1)
# -----------------------
window = 1
training_data = []

for sentence in corpus:
    for i, center in enumerate(sentence):
        context = []
        for j in range(max(0, i-window), min(len(sentence), i+window+1)):
            if j != i:
                context.append(sentence[j])
        for c in context:
            training_data.append((center, c))

# -----------------------
# Initialize weights
# -----------------------
embedding_dim = 5
W1 = np.random.rand(V, embedding_dim)
W2 = np.random.rand(embedding_dim, V)
learning_rate = 0.01

# -----------------------
# Training loop (very small example, simple gradient ascent)
# -----------------------
def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

for epoch in range(100):
    loss = 0
    for center, context in training_data:
        x = one_hot(center)
        h = W1.T @ x                 # hidden layer
        u = W2.T @ h                 # output layer
        y_pred = softmax(u)
        y_true = one_hot(context)
        e = y_pred - y_true           # error
        W2 -= learning_rate * np.outer(h, e)
        W1 -= learning_rate * np.outer(x, W2 @ e)
        loss += -np.log(y_pred[word2idx[context]])
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, loss={loss:.4f}")

# Embeddings are rows of W1
print("\nWord embeddings:")
for word in word2idx:
    print(f"{word}: {W1[word2idx[word]]}")
```

**Notes:**

* This is **manual and small-scale** for illustration.
* Real Word2Vec uses **negative sampling**, **larger embeddings**, **stochastic gradient descent**, and **optimized C implementation**.

---

## 6. Comparison with Built-in Python Method

```python
from gensim.models import Word2Vec

# Train Word2Vec (skip-gram)
model = Word2Vec(sentences=corpus, vector_size=5, window=1, min_count=1, sg=1)
print(model.wv['learning'])
```

**Differences:**

| Feature         | Manual Implementation | Gensim Word2Vec                         |
| --------------- | --------------------- | --------------------------------------- |
| Efficiency      | Very slow             | Highly optimized (C + numpy)            |
| Techniques used | Full softmax          | Negative sampling, hierarchical softmax |
| Scalability     | Tiny corpora only     | Millions of words                       |
| Ease of use     | Long code             | Simple API                              |
| Result quality  | Poor (toy)            | High-quality embeddings                 |

---

## 7. Real-world Python Example

```python
# Real-world: Word similarity
words = ['king', 'queen', 'man', 'woman']
similarity = model.wv.similarity('king', 'queen')
print("Similarity between king and queen:", similarity)

# Analogy: king - man + woman ≈ queen
result = model.wv.most_similar(positive=['king', 'woman'], negative=['man'], topn=1)
print("king - man + woman ->", result)
```

---

## 8. Related Original Articles

1. **Mikolov, T., Chen, K., Corrado, G., & Dean, J. (2013).** Efficient Estimation of Word Representations in Vector Space. arXiv:1301.3781
2. **Mikolov, T., Sutskever, I., Chen, K., Corrado, G., & Dean, J. (2013).** Distributed Representations of Words and Phrases and their Compositionality. NIPS 2013
3. **Mikolov, T., et al. (2013).** Linguistic Regularities in Continuous Space Word Representations. NAACL 2013

---

✅ **Summary:**
Word2Vec converts words into **dense vector embeddings**, capturing **semantic similarity and word relations**. It is **mathematically grounded**, widely used in **NLP tasks**, and can be **manually implemented** for educational purposes, but real applications rely on optimized libraries like **Gensim**.

---

I can also **create a ready-to-run Jupyter Notebook** for Word2Vec including **manual implementation, Gensim comparison, real-world example, mathematics, and references**, similar to the BoW notebook I made earlier.

Do you want me to do that next?
